# README: BERTopic FinLang/investopedia_embedding Variant - Comprehensive Outputs & Tuning Pipeline (GSIB)

## Purpose
Complete BERTopic pipeline using **FinLang/investopedia_embedding embeddings** (domain-specialized for financial text) instead of generic sentence transformers. Combines rigorous hyperparameter tuning via rolling time-series cross-validation with multi-seed robustness evaluation, extended metrics computation, diagnostic visualization, and comprehensive artifact export for thesis documentation and baseline comparison.

## Key Differentiator
- **Embedding Model**: `FinLang/investopedia_embedding` (financial domain-specialized) instead of generic `all-MiniLM-L6-v2`
- **Use Case**: Evaluate whether domain-specific embeddings improve topic quality on financial news corpus compared to generic embeddings
- **Dataset**: Same GSIB dataset (~90,000 unique headlines after deduplication)

## Input Data
- **Source**: `Data/GSIB_model_input.csv`
- **Required Columns**: `headline` (text), `date` (timestamp)
- **Dataset Scale**: ~90,000 unique headlines after deduplication
- **Processing**: Chronologically sorted, deduplicated headlines (exact matches across tickers removed)

## Pipeline Steps

### 1. Data Preparation & Document-Based Temporal Splitting
- **Input**: Raw CSV with headlines and dates
- **Steps**:
  - Parse dates and sort chronologically
  - Remove rows with missing headline or date
  - Deduplicate exact headline matches (prevent ticker-based repeats from dominating topics)
  - Create three **temporal, non-overlapping splits by document count** (70% train, 15% validation, 15% test):
    - Find calendar day boundaries where cumulative document counts align with target percentages
    - **Train**: earliest 70% of documents (by calendar days)
    - **Validation**: middle 15% of documents (by calendar days, kept separate for evaluation)
    - **Test**: latest 15% of documents (by calendar days, held out for final reporting)
  - All documents on the same calendar day stay together in the same split to maintain chronological integrity

### 2. Financial Domain Embedding Generation
- **Model**: `FinLang/investopedia_embedding`
- **Advantages over generic embeddings**: Tuned for financial vocabulary and domain-specific semantics (e.g., "earnings," "dividend," "volatility")
- **Device**: CUDA GPU if available, else CPU
- **Scope**: Generate embeddings only for training split (used for both initial fit and final fit during tuning)

### 3. Initial BERTopic Model Training (Full Train Split)
- **Architecture**:
  - UMAP for dimensionality reduction (15 neighbors, 10 components, cosine metric)
  - HDBSCAN for clustering (min_cluster_size=10, EOM selection method)
  - OnlineCountVectorizer for topic vocabulary (unigrams, custom stopwords including placeholder tokens)
  - FinBERT embeddings from step 2
- **Output**: Topic assignments for training documents, topic word lists with financial domain context

### 4. Validation Split Evaluation & Baseline Metrics
- **Action**: Project validation documents into trained topic space (no model retraining)
- **Metrics Computed** (train/val):
  - Semantic coherence: C_v and NPMI coherence
  - Topic diversity: Unique vocabulary fraction
  - Embedding-based metrics: Intra-topic similarity (within-cluster cosine), inter-topic similarity (between-cluster centroid cosine)
  - Silhouette score (robust variant excluding outliers and singletons)
  - Outlier ratio (fraction of documents assigned to label -1)

### 5. Metric Infrastructure & Coherence Computation
- **Define** helper functions for:
  - Text preprocessing (tokenization, n-gram support, stopword filtering)
  - Topic diversity calculation
  - Robust silhouette scoring (handles outliers, singleton clusters, edge cases)
  - Intra/inter-topic similarity in embedding space
  - Coherence scoring (C_v, NPMI via Gensim, with fallback and numerical stability controls)

### 6. Hyperparameter Tuning with Rolling Time-Series Cross-Validation
- **Search Space**: 36 targeted parameter combinations (reduced to manage computational complexity on the large 90k-document dataset)
  - `n_neighbors` (UMAP): [15, 30, 50]
  - `n_components` (UMAP): [5, 10]
  - `min_cluster_size` (HDBSCAN): [10, 15, 20]
  - `min_samples` (HDBSCAN): [1, 3]
  - `ngram_range` (vectorizer): [(1, 2)]
- **Cross-Validation**: 3-fold rolling/expanding windows on train+validation pool (test remains untouched)
  - Each fold: Train on earlier days, validate on held-out later days
  - Fixed random seed (42) for deterministic model selection
- **Ranking Metric**: Equal-weighted composite score from 5 dimensions:
  - C_v coherence (higher better)
  - NPMI coherence (higher better)
  - Topic diversity (higher better)
  - Intra-topic similarity (higher better)
  - Inter-topic separation (lower inter-topic similarity is better, inverted during normalization)
  - All metrics min-max normalized to [0,1], then averaged
- **Best Model Selection**: Highest composite score configuration selected; fold-wise best configs also reported for temporal stability inspection

### 7. Final Model & Multi-Seed Test Evaluation
- **Training Data**: Refit best hyperparameters on combined train+validation split using FinBERT embeddings
- **Test Evaluation**: Apply final model to held-out test split
- **Robustness Analysis**: Re-fit and evaluate across 9 different random seeds [42, 7, 123, 2024, 99, 11, 77, 314, 2718]:
  - Each seed produces independent UMAP/HDBSCAN initialization
  - Compute all metrics on test split
  - Report mean ± std across seeds to quantify variability
- **Test Metrics** (reported mean ± std):
  - Coherence (C_v, NPMI)
  - Topic diversity
  - Intra/inter-topic similarity
  - Silhouette score
  - Outlier ratio
  - Topic count
- **Thesis-Ready Tables**: Generate formatted reporting tables with results across seeds

### 8. Diagnostics & Extended Visualization
- **Cluster Size Distribution**: Bar chart of topic sizes (excluding outliers)
- **2D Embedding Projection**: UMAP projection of test embeddings colored by point type (outlier/singleton/valid)
- **Seed-Level Variability**: Bar plots showing how outlier ratio and topic count vary across seeds
- **Tuning Leaderboard**: Top configurations from cross-validation ranked by composite score
- **Per-Fold Best Params**: Inspect temporal stability of best configurations across CV folds

### 9. Comprehensive Export & Artifact Preservation
- **Output Directory**: `Outputs/GSIB_BERTopic_FinBert/<run_tag>/` (timestamped, distinct from generic variant)
- **Subdirectories**:
  - `tables/`: Topic info, topic terms (long format), evaluation metrics, tuning results, leaderboard, seed-wise results
  - `row_level/`: Row-wise topic assignments with probabilities
  - `models/`: BERTopic model artifact with FinBERT embeddings
  - `figures/`: HTML Plotly figures (heatmap, hierarchy, topics-over-time, diagnostics, seed variability plots)
  - `meta/`: JSON export summary and detailed export log
- **Export Artifacts**:
  - Topic information table (topics, top terms, document counts)
  - Topic terms in long format (topic ID, word, weight)
  - Row-level assignments (document ID, assigned topic, probability)
  - Tuning results (all parameter combinations with aggregated CV metrics)
  - Tuning leaderboard (top-ranked hyperparameter configurations)
  - Seed-wise final evaluation results (per seed metrics on test set)
  - Summary statistics (mean ± std of final metrics across seeds)
  - Thesis-ready reporting table (formatted for direct inclusion in thesis)
  - Diagnostic figures (embeddings projection, cluster sizes, seed variability)
  - Best hyperparameters (JSON with selected values for reproducibility)
  - Export metadata (run tag, document counts, saved items, notes)

## Key Design Decisions
- **Financial Domain Embedding**: FinBERT provides domain-specific semantic understanding for financial news topics
- **Document-Based Splitting**: Ensures representative train/val/test splits; respects chronological order
- **Temporal Splitting**: Prevents look-ahead bias; respects chronological order of financial news
- **Fixed Seed for Tuning**: Ensures deterministic hyperparameter selection; randomness reserved for robustness analysis
- **Multi-Metric Ranking**: Avoids manual weighting; treats coherence, diversity, and geometry equally
- **Test Set Touch Rule**: Test data only used for final reporting (never in hyperparameter selection or early evaluation)
- **Robust Metrics**: Silhouette, outlier handling, and fallback mechanisms for edge cases (e.g., single valid cluster)
- **Comprehensive Export**: Preserves all intermediate results, metrics, and artifacts for reproducibility and thesis documentation

## Comparative Analysis
- **Compare with**: `04_03_pipeline_BERTopic_TEST_TRAIN_GSIB_outputs.ipynb` (same hyperparameters, generic embeddings)
- **Expected Insights**: Evaluate whether FinBERT embeddings produce:
  - Different topic distributions or granularity
  - More coherent topics in financial domain
  - More stable topic assignments across seeds
  - Better alignment with financial concepts vs. generic language patterns

## Outputs & Next Steps
- Use exported tuning leaderboard to understand hyperparameter sensitivity across metrics
- Compare FinBERT results side-by-side with generic embedding variant for thesis comparison table
- Review fold-wise best configurations to assess temporal stability of selected hyperparameters
- Extract thesis-ready tables from seed-wise results (mean ± std reporting)
- Use exported topic assignments to trace topics back to original documents
- Integrate visualization figures and metrics tables directly into thesis manuscript with explicit FinBERT attribution

Checks Python, Torch, and CUDA/GPU environment availability.

In [1]:
import sys, torch
print('python_executable:', sys.executable)
print('torch_version:', torch.__version__)
print('torch_cuda_runtime:', torch.version.cuda)
print('cuda_available:', torch.cuda.is_available())
print('gpu_count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))

python_executable: c:\Users\gianf\AppData\Local\Programs\Python\Python313\python.exe
torch_version: 2.11.0+cu128
torch_cuda_runtime: 12.8
cuda_available: True
gpu_count: 1
gpu_name: NVIDIA GeForce RTX 3080


Imports core libraries for BERTopic modeling and preprocessing.

In [2]:
import pandas as pd
import numpy as np
import torch
from umap import UMAP
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loads the GSIB model input data and previews basic shape/head.

In [3]:
df = pd.read_csv('Data/GSIB_model_input.csv')
print('Loaded: Data/GSIB_model_input.csv')
print('Shape:', df.shape)
display(df.head())

Loaded: Data/GSIB_model_input.csv
Shape: (121186, 5)


,stock,date,date_only,headline,headline_raw
0,NDAQ,2014-02-17 05:11:00+00:00,2014-02-17,Ameriabank Becomes Market Maker of IFC and EBR...,Ameriabank Becomes Market Maker of IFC and EBR...
1,NDAQ,2014-02-21 05:35:00+00:00,2014-02-21,9th Issue of Corporate Bonds By National Mortg...,9th Issue of Corporate Bonds By National Mortg...
2,NDAQ,2014-04-23 12:39:00+00:00,2014-04-23,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...
3,NDAQ,2014-05-15 10:23:00+00:00,2014-05-15,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...
4,NDAQ,2014-05-26 05:59:00+00:00,2014-05-26,Armswissbank Becomes Market Maker of the 10th ...,Armswissbank Becomes Market Maker of the 10th ...


Parses dates, deduplicates headlines, and creates chronological train/validation/test splits.

In [4]:
# 1. Parse date and prepare chronological data
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'headline']).sort_values('date').reset_index(drop=True)

# Deduplicate semantically identical headlines (topic modeling should not overcount copies across tickers)
before_dedup = len(df)
df = df.drop_duplicates(subset=['headline']).sort_values('date').reset_index(drop=True)
after_dedup = len(df)

# 2. Define target split proportions (by document count, not date count)
SPLIT_TRAIN = 0.70
SPLIT_VAL = 0.15
SPLIT_TEST = 0.15

# 3. Calculate document-based boundaries (keep same day together for chronological integrity)
# Strategy: Find date boundaries such that document counts are close to target percentages
unique_dates = np.array(sorted(df['date'].dt.floor('D').unique()))
day_code, _ = pd.factorize(df['date'].dt.floor('D'), sort=True)

# Find date indices where cumulative document count reaches target percentages
cumsum_docs = np.zeros(len(unique_dates) + 1)
for day_idx, unique_day in enumerate(unique_dates):
    n_docs_in_day = (day_code == day_idx).sum()
    cumsum_docs[day_idx + 1] = cumsum_docs[day_idx] + n_docs_in_day

total_docs = cumsum_docs[-1]
target_train_docs = total_docs * SPLIT_TRAIN
target_val_docs = total_docs * SPLIT_VAL

# Find date boundary indices
train_end_idx = np.searchsorted(cumsum_docs, target_train_docs, side='right') - 1
val_end_idx = np.searchsorted(cumsum_docs, target_train_docs + target_val_docs, side='right') - 1

# Ensure valid indices
train_end_idx = max(1, min(train_end_idx, len(unique_dates) - 2))
val_end_idx = max(train_end_idx + 1, min(val_end_idx, len(unique_dates) - 1))

# 4. Create masks based on date boundaries
mask_train = day_code < train_end_idx
mask_val = (day_code >= train_end_idx) & (day_code < val_end_idx)
mask_test = day_code >= val_end_idx

# 5. Extract splits
train_df = df[mask_train].copy()
val_df = df[mask_val].copy()
test_df = df[mask_test].copy()

# Final lists for the model
train_docs, train_timestamps = train_df['headline'].tolist(), train_df['date'].tolist()
val_docs, val_timestamps = val_df['headline'].tolist(), val_df['date'].tolist()
test_docs, test_timestamps = test_df['headline'].tolist(), test_df['date'].tolist()

print(f'Deduplication: {before_dedup} -> {after_dedup} rows')
print(f"Train size: {len(train_docs)} ({100*len(train_docs)/after_dedup:.1f}%)")
print(f"Validation size: {len(val_docs)} ({100*len(val_docs)/after_dedup:.1f}%)")
print(f"Test size: {len(test_docs)} ({100*len(test_docs)/after_dedup:.1f}%)")
print('Date ranges:')
print(f"  Train: {train_df['date'].min()} -> {train_df['date'].max()}")
print(f"  Val:   {val_df['date'].min()} -> {val_df['date'].max()}")
print(f"  Test:  {test_df['date'].min()} -> {test_df['date'].max()}")

Deduplication: 121186 -> 90048 rows
Train size: 62969 (69.9%)
Validation size: 13503 (15.0%)
Test size: 13576 (15.1%)
Date ranges:
  Train: 2014-02-17 05:11:00+00:00 -> 2025-06-16 22:56:30+00:00
  Val:   2025-06-17 02:28:37+00:00 -> 2025-11-18 23:36:34+00:00
  Test:  2025-11-19 00:09:00+00:00 -> 2026-04-11 17:07:00+00:00


Configures embedding model, stopwords, and computes train embeddings.

In [7]:


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Add placeholder tokens to stopwords so they do not dominate topics
placeholder_stopwords = {'__ticker__', '__company__', 'ticker', 'company'}
model_stopwords = sorted(set(ENGLISH_STOP_WORDS).union(placeholder_stopwords))

sentence_model = SentenceTransformer('FinLang/finance-embeddings-investopedia', device=device)
train_embeddings = sentence_model.encode(
    train_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
print(f'Embeddings computed on device: {device}')

Torch CUDA available: True
GPU: NVIDIA GeForce RTX 3080


Batches: 100%|██████████| 984/984 [00:28<00:00, 34.69it/s]


Embeddings computed on device: cuda


Builds and fits BERTopic on the training split.

In [8]:
print("Training BERTopic model on training split...\n")

umap_model = UMAP(n_neighbors=15, n_components=10, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = OnlineCountVectorizer(stop_words=model_stopwords)

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  calculate_probabilities=True,
  verbose=True
)

# Train BERTopic only on train split
topics_train, probs_train = topic_model.fit_transform(train_docs, embeddings=train_embeddings)

print("\nModel training complete!")

2026-04-22 22:26:26,177 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic model on training split...



2026-04-22 22:26:51,721 - BERTopic - Dimensionality - Completed ✓
2026-04-22 22:26:51,723 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-22 23:09:39,931 - BERTopic - Cluster - Completed ✓
2026-04-22 23:09:39,945 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-22 23:09:40,626 - BERTopic - Representation - Completed ✓



Model training complete!


Displays BERTopic topic summary table.

In [9]:
# Get topic information
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,24283,-1_goldman_banks_bank_billion,"[goldman, banks, bank, billion, street, bankin...",[Hong Kong Put Pressure on Three Major Banks t...
1,0,799,0_bitcoin_spot_etf_ethereum,"[bitcoin, spot, etf, ethereum, grayscale, sec,...",[UPDATE 1-BlackRock files for spot ethereum ET...
2,1,577,1_appoints_names_head_hires,"[appoints, names, head, hires, officer, chief,...","[__COMPANY__ names new corporate, investment b..."
3,2,496,2_oil_opec_supply_steadies,"[oil, opec, supply, steadies, demand, weekly, ...",[Oil Holds Biggest Gain Since March Ahead of O...
4,3,493,3_according_blue_chip_10,"[according, blue, chip, 10, best, undervalued,...",[12 Best Blue Chip Stocks To Invest In Accordi...
...,...,...,...,...,...
888,887,10,887_repurchases_80m_repurchase_400m,"[repurchases, 80m, repurchase, 400m, 37m, plan...",[__COMPANY__ __TICKER__ Announces 80M Share Re...
889,888,10,888_bouncebit_pilots_coinbase_buidl,"[bouncebit, pilots, coinbase, buidl, kraken, c...","[BlackRock files for Bitcoin ETF, Coinbase and..."
890,889,10,889_managing_directors_334_155,"[managing, directors, 334, 155, promotes, 85, ...",[UPDATE 1- __COMPANY__ appoints 334 new managi...
891,890,10,890_ck_hutchison_ports_port,"[ck, hutchison, ports, port, panama, angered, ...",[Doubts over CK Hutchison port deal add to con...


Projects validation documents into trained topic space.

In [10]:
bertopic_topics = [
    [word for word, _ in topic_model.get_topic(t)]
    for t in topic_model.get_topics().keys() if t != -1
    if topic_model.get_topic(t) is not None
 ]

# Project validation documents into trained topic space
val_topics, val_probs = topic_model.transform(val_docs)

print("Validation transform complete (test set remains untouched).")

Batches: 100%|██████████| 422/422 [00:07<00:00, 58.38it/s]
2026-04-22 23:09:50,083 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-22 23:10:04,445 - BERTopic - Dimensionality - Completed ✓
2026-04-22 23:10:04,446 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-22 23:10:06,165 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-22 23:18:47,961 - BERTopic - Probabilities - Completed ✓
2026-04-22 23:18:47,961 - BERTopic - Cluster - Completed ✓


Validation transform complete (test set remains untouched).


Defines evaluation utilities and computes coherence/diversity/similarity metrics.

In [11]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import nltk
import warnings
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('stopwords', quiet=True)

# Numerical stability controls
COHERENCE_EPS = 1e-8
COHERENCE_MIN_TEXTS = 20
COHERENCE_MIN_VOCAB = 30
SIMILARITY_MAX_POINTS_PER_TOPIC = 300

# Build a tokenizer aligned with BERTopic vocabulary (supports unigrams + bigrams)
stop_words = list(model_stopwords) if 'model_stopwords' in globals() else list(stopwords.words('english'))
coherence_vectorizer = CountVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b\w\w+\b'
)
coherence_analyzer = coherence_vectorizer.build_analyzer()

def preprocess_documents(doc_list):
    tokenized = [coherence_analyzer(str(doc).lower()) for doc in doc_list]
    return [tokens for tokens in tokenized if len(tokens) >= 2]

processed_train_docs = preprocess_documents(train_docs)
processed_val_docs = preprocess_documents(val_docs)

# Dictionary from train split only
id2word = Dictionary(processed_train_docs)

def topic_diversity(topics):
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    return len(unique_words) / len(all_words) if len(all_words) > 0 else 0.0

def _filter_topics_for_dictionary(topics, dictionary, min_words_per_topic=3):
    vocab = dictionary.token2id
    filtered = []
    for topic in topics:
        cleaned = []
        for term in topic:
            token = str(term).strip().lower()
            if token in vocab:
                cleaned.append(token)
        cleaned = list(dict.fromkeys(cleaned))
        if len(cleaned) >= min_words_per_topic:
            filtered.append(cleaned)
    return filtered

def safe_silhouette_score(
    embeddings,
    labels,
    outlier_label=-1,
    metric='cosine',
    min_cluster_size_for_metric=2,
    min_points_for_metric=10,
    drop_singletons=True
):
    if embeddings is None or labels is None:
        return np.nan

    labels_arr = np.asarray(labels)
    embeddings_arr = np.asarray(embeddings)

    if embeddings_arr.ndim != 2 or labels_arr.ndim != 1:
        return np.nan
    if embeddings_arr.shape[0] != labels_arr.shape[0]:
        return np.nan

    finite_mask = np.isfinite(labels_arr.astype(float, copy=False))
    valid_mask = (labels_arr != outlier_label) & finite_mask
    if valid_mask.sum() < min_points_for_metric:
        return np.nan

    labels_valid = labels_arr[valid_mask]
    emb_valid = embeddings_arr[valid_mask]

    if drop_singletons:
        unique_labels, counts = np.unique(labels_valid, return_counts=True)
        keep_labels = unique_labels[counts >= min_cluster_size_for_metric]
        if len(keep_labels) < 2:
            return np.nan
        keep_mask = np.isin(labels_valid, keep_labels)
        labels_valid = labels_valid[keep_mask]
        emb_valid = emb_valid[keep_mask]

    unique_labels, counts = np.unique(labels_valid, return_counts=True)
    if len(unique_labels) < 2:
        return np.nan
    if np.any(counts < min_cluster_size_for_metric):
        return np.nan
    if len(labels_valid) < max(min_points_for_metric, len(unique_labels) + 1):
        return np.nan

    try:
        value = float(silhouette_score(emb_valid, labels_valid, metric=metric))
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan

def intra_inter_topic_similarity(
    embeddings,
    labels,
    outlier_label=-1,
    max_points_per_topic=SIMILARITY_MAX_POINTS_PER_TOPIC,
    random_state=42
):
    if embeddings is None or labels is None:
        return np.nan, np.nan

    labels_arr = np.asarray(labels)
    embeddings_arr = np.asarray(embeddings)

    if embeddings_arr.ndim != 2 or labels_arr.ndim != 1:
        return np.nan, np.nan
    if embeddings_arr.shape[0] != labels_arr.shape[0]:
        return np.nan, np.nan

    finite_mask = np.isfinite(labels_arr.astype(float, copy=False))
    valid_mask = (labels_arr != outlier_label) & finite_mask
    if valid_mask.sum() < 2:
        return np.nan, np.nan

    labels_valid = labels_arr[valid_mask]
    emb_valid = embeddings_arr[valid_mask]
    rng = np.random.default_rng(random_state)

    intra_values = []
    centroids = []

    for topic_label in np.unique(labels_valid):
        topic_mask = labels_valid == topic_label
        topic_emb = emb_valid[topic_mask]
        if topic_emb.shape[0] == 0:
            continue

        if topic_emb.shape[0] > max_points_per_topic:
            sampled_idx = rng.choice(topic_emb.shape[0], size=max_points_per_topic, replace=False)
            topic_emb = topic_emb[sampled_idx]

        centroids.append(topic_emb.mean(axis=0))

        if topic_emb.shape[0] >= 2:
            sim_matrix = cosine_similarity(topic_emb)
            upper_idx = np.triu_indices(sim_matrix.shape[0], k=1)
            if len(upper_idx[0]) > 0:
                intra_values.append(float(np.nanmean(sim_matrix[upper_idx])))

    intra_topic = float(np.nanmean(intra_values)) if len(intra_values) > 0 else np.nan

    if len(centroids) < 2:
        inter_topic = np.nan
    else:
        centroids_arr = np.vstack(centroids)
        centroid_sim = cosine_similarity(centroids_arr)
        upper_idx = np.triu_indices(centroid_sim.shape[0], k=1)
        inter_topic = float(np.nanmean(centroid_sim[upper_idx])) if len(upper_idx[0]) > 0 else np.nan

    return intra_topic, inter_topic

def coherence_score(topics, texts, dictionary, coherence_type='c_v', jitter=COHERENCE_EPS):
    clean_texts = [t for t in texts if isinstance(t, list) and len(t) >= 2]
    if len(clean_texts) < COHERENCE_MIN_TEXTS or len(dictionary) < COHERENCE_MIN_VOCAB:
        return np.nan if jitter <= 0 else float(jitter)

    local_dict = Dictionary(clean_texts)
    local_dict.filter_extremes(no_below=2, no_above=0.95)
    local_dict.compactify()
    if len(local_dict) < 20:
        return np.nan if jitter <= 0 else float(jitter)

    filtered_topics = _filter_topics_for_dictionary(topics, local_dict, min_words_per_topic=3)
    if len(filtered_topics) < 2:
        return np.nan if jitter <= 0 else float(jitter)

    def _compute_value(kind):
        if kind == 'u_mass':
            corpus = [local_dict.doc2bow(doc) for doc in clean_texts]
            cm = CoherenceModel(
                topics=filtered_topics,
                corpus=corpus,
                dictionary=local_dict,
                coherence='u_mass'
            )
        else:
            cm = CoherenceModel(
                topics=filtered_topics,
                texts=clean_texts,
                dictionary=local_dict,
                coherence=kind
            )
        return float(cm.get_coherence())

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning, module='gensim')
        with np.errstate(divide='ignore', invalid='ignore'):
            try:
                value = _compute_value(coherence_type)
                if np.isfinite(value):
                    return float(value)
            except Exception:
                pass

            # Only c_v gets u_mass fallback. For c_npmi we avoid mixed-scale fallback.
            if coherence_type == 'c_v':
                try:
                    fallback_value = _compute_value('u_mass')
                    if np.isfinite(fallback_value):
                        return float(fallback_value)
                except Exception:
                    pass

    return np.nan if jitter <= 0 else float(jitter)

cv_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_v')
npmi_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_npmi')
cv_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_v')
npmi_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_npmi')
div_train = topic_diversity(bertopic_topics)
val_outlier_ratio = float(np.mean(np.array(val_topics) == -1)) if len(val_topics) > 0 else np.nan
valid_topic_count = len(_filter_topics_for_dictionary(bertopic_topics, id2word, min_words_per_topic=3))

val_embeddings_for_metrics = sentence_model.encode(
    val_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
sil_val = safe_silhouette_score(val_embeddings_for_metrics, val_topics)
sil_val = 0.0 if not np.isfinite(sil_val) else sil_val
intra_val, inter_val = intra_inter_topic_similarity(val_embeddings_for_metrics, val_topics)

print(f"BERTopic Coherence C_v (train): {cv_train:.4f}")
print(f"BERTopic Coherence NPMI (train): {npmi_train:.4f}")
print(f"BERTopic Coherence C_v (val):   {cv_val:.4f}")
print(f"BERTopic Coherence NPMI (val):   {npmi_val:.4f}")
print(f"BERTopic Topic Diversity (train): {div_train:.4f}")
print(f"Validation outlier ratio (-1 topics): {val_outlier_ratio:.4f}")
print(f"Validation silhouette (cosine, no outliers): {sil_val:.4f}")
print(f"Validation intra-topic similarity: {intra_val:.4f}")
print(f"Validation inter-topic similarity: {inter_val:.4f}")
print(f"Valid topic count for coherence: {valid_topic_count}")

Batches: 100%|██████████| 211/211 [00:06<00:00, 32.82it/s]


BERTopic Coherence C_v (train): 0.4983
BERTopic Coherence NPMI (train): 0.0713
BERTopic Coherence C_v (val):   0.3660
BERTopic Coherence NPMI (val):   -0.2281
BERTopic Topic Diversity (train): 0.6041
Validation outlier ratio (-1 topics): 0.5369
Validation silhouette (cosine, no outliers): 0.1038
Validation intra-topic similarity: 0.6044
Validation inter-topic similarity: 0.2990
Valid topic count for coherence: 892


## Tuning Strategy & Composite Scoring Explained

This cell runs a **comprehensive hyperparameter search** with a Jehnen-style multi-metric ranking setup.

### Search Design
- **Reduced Parameter Space**: 60 combinations instead of 900 (15x faster)
- **Folds**: 3 rolling/expanding time-series CV folds on train+val (test untouched).
- **Seed**: Fixed `RANDOM_SEED=42` for deterministic model selection.

### Per-Fold Metrics Used for Ranking (Equal Weighting)
| Metric | Direction | Notes |
|--------|-----------|-------|
| **c_v coherence** | Higher is better | Semantic coherence of topic words |
| **NPMI coherence** | Higher is better | Strong coherence signal for topic quality |
| **Topic diversity** | Higher is better | Fraction of unique words across topics |
| **Intra-topic similarity** | Higher is better | Cohesion within topic clusters in embedding space |
| **Inter-topic similarity** | Lower is better | Separation between topic clusters in embedding space |

### Aggregation & Normalization
1. Average each metric across folds per hyperparameter combination.
2. Min-max normalize each metric to [0,1].
3. Invert inter-topic similarity during normalization so lower raw inter-topic similarity gets higher normalized score.

### Composite Scoring Formula (No Manual Weighting)
All five ranking metrics are weighted equally:

$$
\text{score} = \frac{1}{5}\left(
\text{c\_v}_{\text{norm}} + \text{npmi}_{\text{norm}} + \text{diversity}_{\text{norm}} + \text{intra}_{\text{norm}} + \text{inter\_separation}_{\text{norm}}
\right)
$$

This is equivalent to **no preferential weighting** between ranking metrics.

### Model Selection
The highest composite score row is selected as `best_params`. Fold-level best rows are also reported to inspect temporal stability.

### Next: Multi-Seed Final Evaluation
After selecting best params, the final model is re-fit and evaluated across 9 random seeds on the untouched test set.

Runs rolling CV grid search and selects best BERTopic configuration.

In [12]:
from itertools import product
import time

# ================================================================================
# HYPERPARAMETER TUNING: GRID SEARCH + ROLLING CV + JEHNEN-STYLE EQUAL-WEIGHT RANKING
# ================================================================================
# Ranking metrics (equal weight):
#   - c_v coherence
#   - c_npmi coherence
#   - topic diversity
#   - intra-topic similarity (higher better)
#   - inter-topic similarity (lower better; inverted during normalization)
# ================================================================================

# -------------------- Time-series CV controls (fixed seed for model selection) --------------------
N_FOLDS = 3  # rolling/expanding buckets before test
RANDOM_SEED = 42

# Build tuning pool from train + validation only (test remains untouched)
tune_df = pd.concat([train_df, val_df], axis=0).sort_values('date').reset_index(drop=True)
tune_docs = tune_df['headline'].tolist()
tune_timestamps = tune_df['date'].tolist()

# Precompute embeddings for the full tuning pool once
tune_embeddings = sentence_model.encode(
    tune_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

def sanitize_metric(value, fallback=np.nan):
    try:
        v = float(value)
    except Exception:
        return fallback
    return v if np.isfinite(v) else fallback

def make_expanding_folds(df_dates, n_folds=3, embargo_n=1):
    unique_days = np.array(sorted(pd.to_datetime(df_dates).dt.floor('D').unique()))
    n_days = len(unique_days)
    edges = np.linspace(0, n_days, n_folds + 2, dtype=int)
    day_code, _ = pd.factorize(pd.to_datetime(df_dates).dt.floor('D'), sort=True)

    folds = []
    for fold_idx in range(n_folds):
        train_end = edges[fold_idx + 1]
        val_start = train_end + embargo_n
        val_end = edges[fold_idx + 2]

        if train_end <= 0 or val_start >= val_end:
            continue

        train_mask = day_code < train_end
        val_mask = (day_code >= val_start) & (day_code < val_end)

        train_idx = np.where(train_mask)[0]
        val_idx = np.where(val_mask)[0]

        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        folds.append({
            'fold': fold_idx + 1,
            'train_idx': train_idx,
            'val_idx': val_idx,
            'train_start': unique_days[0],
            'train_end': unique_days[train_end - 1],
            'val_start': unique_days[val_start],
            'val_end': unique_days[val_end - 1]
        })

    return folds

folds = make_expanding_folds(tune_df['date'], n_folds=N_FOLDS)
print(f'Generated {len(folds)} rolling folds (train+val pool only).')
for fold in folds:
    print(
        f"Fold {fold['fold']}: train {fold['train_start']} -> {fold['train_end']} | "
        f"val {fold['val_start']} -> {fold['val_end']}"
    )

param_grid = {
    'n_neighbors': [15, 30, 50],           # Reduced from [10, 30, 50, 70, 90]
    'n_components': [5, 10],                # Reduced from [4, 6, 8]
    'min_cluster_size': [10, 15, 20],       # Reduced from [8, 12, 16, 24, 32]
    'min_samples': [1, 3],                  # Reduced from [1, 2, 4, 6]
    'ngram_range': [(1, 2)]
}

search_space = list(product(
    param_grid['n_neighbors'],
    param_grid['n_components'],
    param_grid['min_cluster_size'],
    param_grid['min_samples'],
    param_grid['ngram_range']
))
expected_combinations = (
    len(param_grid['n_neighbors'])
    * len(param_grid['n_components'])
    * len(param_grid['min_cluster_size'])
    * len(param_grid['min_samples'])
    * len(param_grid['ngram_range'])
)

print(f'Configured combinations: {expected_combinations} (target ~300)')
print(f'Running {len(search_space)} parameter combinations x {len(folds)} folds (fixed seed={RANDOM_SEED})')

cv_rows = []

for trial, (n_neighbors, n_components, min_cluster_size, min_samples, ngram_range) in enumerate(search_space, start=1):
    print(
        f"[Trial {trial}/{len(search_space)}] n_neighbors={n_neighbors}, n_components={n_components}, "
        f"min_cluster_size={min_cluster_size}, min_samples={min_samples}, ngram_range={ngram_range}"
    )

    for fold in folds:
        train_idx = fold['train_idx']
        val_idx = fold['val_idx']

        fold_train_docs = [tune_docs[i] for i in train_idx]
        fold_val_docs = [tune_docs[i] for i in val_idx]
        fold_train_embeddings = tune_embeddings[train_idx]
        fold_val_embeddings = tune_embeddings[val_idx]

        processed_train_fold = preprocess_documents(fold_train_docs)
        processed_val_fold = preprocess_documents(fold_val_docs)
        id2word_fold = Dictionary(processed_train_fold)

        start = time.perf_counter()

        umap_model_i = UMAP(
            n_neighbors=n_neighbors,
            n_components=n_components,
            metric='cosine',
            random_state=RANDOM_SEED
        )
        hdbscan_model_i = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True,
            core_dist_n_jobs=-1
        )
        vectorizer_model_i = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=ngram_range)

        topic_model_i = BERTopic(
            embedding_model=sentence_model,
            umap_model=umap_model_i,
            hdbscan_model=hdbscan_model_i,
            vectorizer_model=vectorizer_model_i,
            calculate_probabilities=False,
            verbose=False
        )

        _topics_train_i, _ = topic_model_i.fit_transform(fold_train_docs, embeddings=fold_train_embeddings)
        val_topics_i, _ = topic_model_i.transform(fold_val_docs, embeddings=fold_val_embeddings)

        topic_words_i = [
            [word for word, _ in topic_model_i.get_topic(t)]
            for t in topic_model_i.get_topics().keys()
            if t != -1 and topic_model_i.get_topic(t) is not None
        ]

        has_enough_for_coherence = (
            len(processed_val_fold) >= 20 and
            len(id2word_fold) >= 30 and
            len(topic_words_i) >= 2
        )

        cv_val_i = sanitize_metric(
            coherence_score(topic_words_i, processed_val_fold, id2word_fold, 'c_v', jitter=COHERENCE_EPS),
            fallback=COHERENCE_EPS
        ) if has_enough_for_coherence else COHERENCE_EPS

        npmi_val_i = sanitize_metric(
            coherence_score(topic_words_i, processed_val_fold, id2word_fold, 'c_npmi', jitter=COHERENCE_EPS),
            fallback=COHERENCE_EPS
        ) if has_enough_for_coherence else COHERENCE_EPS

        div_i = sanitize_metric(topic_diversity(topic_words_i), fallback=0.0)
        intra_i, inter_i = intra_inter_topic_similarity(fold_val_embeddings, val_topics_i)
        intra_i = sanitize_metric(intra_i, fallback=0.0)
        inter_i = sanitize_metric(inter_i, fallback=1.0)

        n_topics_i = int(sum(1 for t in topic_model_i.get_topics().keys() if int(t) != -1))
        elapsed = time.perf_counter() - start

        cv_rows.append({
            'trial': trial,
            'fold': fold['fold'],
            'random_seed': RANDOM_SEED,
            'n_neighbors': n_neighbors,
            'n_components': n_components,
            'min_cluster_size': min_cluster_size,
            'min_samples': min_samples,
            'ngram_range': str(ngram_range),
            'n_topics': n_topics_i,
            'cv_val': cv_val_i,
            'npmi_val': npmi_val_i,
            'topic_diversity': div_i,
            'intra_topic_similarity': intra_i,
            'inter_topic_similarity': inter_i,
            'fit_seconds': elapsed
        })

cv_results = pd.DataFrame(cv_rows)

agg_cols = [
    'cv_val', 'npmi_val', 'topic_diversity',
    'intra_topic_similarity', 'inter_topic_similarity',
    'n_topics', 'fit_seconds'
]
tuning_results = cv_results.groupby(
    ['n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
)[agg_cols].mean()

def minmax_norm(series, higher_is_better=True):
    s = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    if s.dropna().empty:
        return pd.Series(0.0, index=s.index)
    filled = s.fillna(s.min() if higher_is_better else s.max())
    min_v, max_v = float(filled.min()), float(filled.max())
    if max_v == min_v:
        return pd.Series(0.0, index=filled.index)
    norm = (filled - min_v) / (max_v - min_v)
    return norm

tuning_results['cv_val_norm'] = minmax_norm(tuning_results['cv_val'], higher_is_better=True)
tuning_results['npmi_val_norm'] = minmax_norm(tuning_results['npmi_val'], higher_is_better=True)
tuning_results['topic_diversity_norm'] = minmax_norm(tuning_results['topic_diversity'], higher_is_better=True)
tuning_results['intra_topic_similarity_norm'] = minmax_norm(tuning_results['intra_topic_similarity'], higher_is_better=True)
tuning_results['inter_topic_separation_norm'] = minmax_norm(tuning_results['inter_topic_similarity'], higher_is_better=False)

# Equal weighting across Jehnen-style ranking metrics (no preferential weighting)
tuning_results['composite_score'] = (
    tuning_results['cv_val_norm'] +
    tuning_results['npmi_val_norm'] +
    tuning_results['topic_diversity_norm'] +
    tuning_results['intra_topic_similarity_norm'] +
    tuning_results['inter_topic_separation_norm']
) / 5.0

tuning_results = tuning_results.sort_values('composite_score', ascending=False).reset_index(drop=True)
best_params = tuning_results.iloc[0].to_dict()

# Bucket-wise (fold-wise) best params to inspect drift over time
fold_agg = cv_results.groupby(
    ['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
)[['cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity', 'n_topics']].mean()

fold_agg['cv_val_norm'] = fold_agg.groupby('fold')['cv_val'].transform(lambda x: minmax_norm(x, True))
fold_agg['npmi_val_norm'] = fold_agg.groupby('fold')['npmi_val'].transform(lambda x: minmax_norm(x, True))
fold_agg['topic_diversity_norm'] = fold_agg.groupby('fold')['topic_diversity'].transform(lambda x: minmax_norm(x, True))
fold_agg['intra_topic_similarity_norm'] = fold_agg.groupby('fold')['intra_topic_similarity'].transform(lambda x: minmax_norm(x, True))
fold_agg['inter_topic_separation_norm'] = fold_agg.groupby('fold')['inter_topic_similarity'].transform(lambda x: minmax_norm(x, False))

fold_agg['fold_score'] = (
    fold_agg['cv_val_norm'] +
    fold_agg['npmi_val_norm'] +
    fold_agg['topic_diversity_norm'] +
    fold_agg['intra_topic_similarity_norm'] +
    fold_agg['inter_topic_separation_norm']
) / 5.0

best_per_fold = (
    fold_agg.sort_values(['fold', 'fold_score'], ascending=[True, False])
            .groupby('fold', as_index=False)
            .head(1)
            .reset_index(drop=True)
)

print('\nTop configurations (overall):')
display(tuning_results.head(10))

print('\nBest params by time bucket (fold):')
display(best_per_fold[['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range', 'cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity', 'fold_score']])

print('\nAverage fit time per model run (seconds):', round(float(cv_results['fit_seconds'].mean()), 2))
print('Best overall configuration selected from rolling CV (fixed-seed model selection):')
display(pd.DataFrame([best_params]))

Batches: 100%|██████████| 1195/1195 [00:33<00:00, 35.80it/s]


Generated 3 rolling folds (train+val pool only).
Fold 1: train 2014-02-17 00:00:00+00:00 -> 2018-06-15 00:00:00+00:00 | val 2018-06-19 00:00:00+00:00 -> 2021-07-20 00:00:00+00:00
Fold 2: train 2014-02-17 00:00:00+00:00 -> 2021-07-20 00:00:00+00:00 | val 2021-07-22 00:00:00+00:00 -> 2023-09-19 00:00:00+00:00
Fold 3: train 2014-02-17 00:00:00+00:00 -> 2023-09-19 00:00:00+00:00 | val 2023-09-21 00:00:00+00:00 -> 2025-11-18 00:00:00+00:00
Configured combinations: 36 (target ~300)
Running 36 parameter combinations x 3 folds (fixed seed=42)
[Trial 1/36] n_neighbors=15, n_components=5, min_cluster_size=10, min_samples=1, ngram_range=(1, 2)
[Trial 2/36] n_neighbors=15, n_components=5, min_cluster_size=10, min_samples=3, ngram_range=(1, 2)
[Trial 3/36] n_neighbors=15, n_components=5, min_cluster_size=15, min_samples=1, ngram_range=(1, 2)
[Trial 4/36] n_neighbors=15, n_components=5, min_cluster_size=15, min_samples=3, ngram_range=(1, 2)
[Trial 5/36] n_neighbors=15, n_components=5, min_cluster_si

,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,n_topics,fit_seconds,cv_val_norm,npmi_val_norm,topic_diversity_norm,intra_topic_similarity_norm,inter_topic_separation_norm,composite_score
0,50,10,20,3,"(1, 2)",0.564609,0.072131,0.810327,0.534300,0.405183,173.666667,88.612120,1.000000,1.000000,0.662219,0.436784,0.606030,0.741007
1,50,10,10,3,"(1, 2)",0.560724,0.059154,0.822662,0.573223,0.361022,312.666667,88.485573,0.913377,0.732780,1.000000,1.000000,0.000000,0.729231
2,50,5,20,3,"(1, 2)",0.563285,0.069557,0.799453,0.525085,0.411121,171.333333,84.824476,0.970478,0.947000,0.364457,0.303440,0.687528,0.654581
3,50,10,15,3,"(1, 2)",0.549180,0.054054,0.815486,0.545791,0.395636,219.666667,86.646218,0.655951,0.627767,0.803478,0.603055,0.475016,0.633053
4,50,5,20,1,"(1, 2)",0.561764,0.067346,0.794907,0.517786,0.417548,197.000000,85.516270,0.936577,0.901482,0.239985,0.197818,0.775726,0.610318
5,15,5,20,3,"(1, 2)",0.557201,0.059637,0.799027,0.508145,0.430560,214.333333,67.839121,0.834824,0.742717,0.352781,0.058317,0.954297,0.588587
6,15,10,20,3,"(1, 2)",0.552622,0.057386,0.801733,0.505182,0.432971,213.333333,67.536024,0.732697,0.696366,0.426881,0.015432,0.987380,0.571751
7,30,5,10,3,"(1, 2)",0.551215,0.047305,0.808343,0.555314,0.378752,339.333333,76.792367,0.701328,0.488772,0.607899,0.740846,0.243312,0.556431
8,30,5,20,3,"(1, 2)",0.549872,0.056667,0.799714,0.516625,0.421290,186.666667,74.479202,0.671383,0.681565,0.371592,0.181021,0.827083,0.546529
9,30,5,15,3,"(1, 2)",0.546867,0.048564,0.808342,0.530278,0.406707,236.333333,75.403831,0.604384,0.514710,0.607862,0.378580,0.626946,0.546496



Best params by time bucket (fold):


,fold,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,fold_score
0,1,50,10,10,3,"(1, 2)",0.671777,0.120649,0.825000,0.536749,0.422032,0.699535
1,2,50,10,20,3,"(1, 2)",0.545100,0.061272,0.837719,0.547655,0.374690,0.774052
2,3,50,10,10,3,"(1, 2)",0.492293,0.021970,0.820606,0.605955,0.313763,0.759159



Average fit time per model run (seconds): 77.9
Best overall configuration selected from rolling CV (fixed-seed model selection):


,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,n_topics,fit_seconds,cv_val_norm,npmi_val_norm,topic_diversity_norm,intra_topic_similarity_norm,inter_topic_separation_norm,composite_score
0,50,10,20,3,"(1, 2)",0.564609,0.072131,0.810327,0.5343,0.405183,173.666667,88.61212,1.0,1.0,0.662219,0.436784,0.60603,0.741007


Refits best config and evaluates seed-wise variability on test split.

In [13]:
from ast import literal_eval

# Refit best hyperparameter configuration on train+val, then evaluate variability across random seeds on test
EVAL_SEEDS = [42, 7, 123, 2024, 99, 11, 77, 314, 2718]

train_val_docs = train_docs + val_docs
train_val_embeddings = sentence_model.encode(
    train_val_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
test_embeddings = sentence_model.encode(
    test_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

processed_train_val_docs = preprocess_documents(train_val_docs)
processed_test_docs = preprocess_documents(test_docs)
id2word_train_val = Dictionary(processed_train_val_docs)

best_ngram = literal_eval(best_params['ngram_range'])
seed_rows = []

for seed in EVAL_SEEDS:
    print(f'Running final fit/eval for seed={seed}...')

    best_umap = UMAP(
        n_neighbors=int(best_params['n_neighbors']),
        n_components=int(best_params['n_components']),
        metric='cosine',
        random_state=seed
    )
    best_hdbscan = HDBSCAN(
        min_cluster_size=int(best_params['min_cluster_size']),
        min_samples=int(best_params['min_samples']),
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    best_vectorizer = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=best_ngram)

    best_topic_model = BERTopic(
        embedding_model=sentence_model,
        umap_model=best_umap,
        hdbscan_model=best_hdbscan,
        vectorizer_model=best_vectorizer,
        calculate_probabilities=True,
        verbose=False
    )

    _topics_train_val, _ = best_topic_model.fit_transform(train_val_docs, embeddings=train_val_embeddings)
    test_topics, _ = best_topic_model.transform(test_docs, embeddings=test_embeddings)

    best_topic_words = [
        [word for word, _ in best_topic_model.get_topic(t)]
        for t in best_topic_model.get_topics().keys()
        if t != -1 and best_topic_model.get_topic(t) is not None
    ]

    cv_test = sanitize_metric(
        coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_v', jitter=COHERENCE_EPS),
        fallback=COHERENCE_EPS
    )
    npmi_test = sanitize_metric(
        coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_npmi', jitter=COHERENCE_EPS),
        fallback=COHERENCE_EPS
    )
    div_test = sanitize_metric(topic_diversity(best_topic_words), fallback=0.0)
    intra_test, inter_test = intra_inter_topic_similarity(test_embeddings, test_topics)
    intra_test = sanitize_metric(intra_test, fallback=0.0)
    inter_test = sanitize_metric(inter_test, fallback=1.0)

    # Final-only diagnostics (not used in tuning)
    sil_test = sanitize_metric(
        safe_silhouette_score(test_embeddings, test_topics, outlier_label=-1, metric='cosine'),
        fallback=0.0
    )
    test_outlier_ratio = float(np.mean(np.array(test_topics) == -1)) if len(test_topics) > 0 else np.nan
    test_topic_count = int(sum(1 for t in best_topic_model.get_topics().keys() if int(t) != -1))

    seed_rows.append({
        'seed': seed,
        'cv_test': cv_test,
        'npmi_test': npmi_test,
        'topic_diversity_test': div_test,
        'intra_topic_similarity_test': intra_test,
        'inter_topic_similarity_test': inter_test,
        'test_silhouette': sil_test,
        'test_outlier_ratio': test_outlier_ratio,
        'test_topic_count': test_topic_count
    })

seed_results = pd.DataFrame(seed_rows)

print('\nPer-seed test metrics:')
display(seed_results)

summary_stats = pd.DataFrame([{
    'cv_test_mean': seed_results['cv_test'].mean(skipna=True),
    'cv_test_std': seed_results['cv_test'].std(skipna=True),
    'npmi_test_mean': seed_results['npmi_test'].mean(skipna=True),
    'npmi_test_std': seed_results['npmi_test'].std(skipna=True),
    'topic_diversity_mean': seed_results['topic_diversity_test'].mean(skipna=True),
    'topic_diversity_std': seed_results['topic_diversity_test'].std(skipna=True),
    'intra_topic_similarity_mean': seed_results['intra_topic_similarity_test'].mean(skipna=True),
    'intra_topic_similarity_std': seed_results['intra_topic_similarity_test'].std(skipna=True),
    'inter_topic_similarity_mean': seed_results['inter_topic_similarity_test'].mean(skipna=True),
    'inter_topic_similarity_std': seed_results['inter_topic_similarity_test'].std(skipna=True),
    'test_silhouette_mean': seed_results['test_silhouette'].mean(skipna=True),
    'test_silhouette_std': seed_results['test_silhouette'].std(skipna=True),
    'test_outlier_ratio_mean': seed_results['test_outlier_ratio'].mean(skipna=True),
    'test_outlier_ratio_std': seed_results['test_outlier_ratio'].std(skipna=True),
    'test_topic_count_mean': seed_results['test_topic_count'].mean(skipna=True),
    'test_topic_count_std': seed_results['test_topic_count'].std(skipna=True)
}])

print('\nFinal test metrics variability across random seeds:')
display(summary_stats)

def format_mean_std(mean_value, std_value, digits=4):
    if not np.isfinite(mean_value):
        return 'NA'
    if not np.isfinite(std_value):
        return f"{mean_value:.{digits}f} ± NA"
    return f"{mean_value:.{digits}f} ± {std_value:.{digits}f}"

thesis_report = pd.DataFrame([{
    'Model': 'BERTopic (best CV config)',
    'Seeds (n)': len(EVAL_SEEDS),
    'Coherence C_v (test)': format_mean_std(summary_stats.at[0, 'cv_test_mean'], summary_stats.at[0, 'cv_test_std'], digits=4),
    'Coherence NPMI (test)': format_mean_std(summary_stats.at[0, 'npmi_test_mean'], summary_stats.at[0, 'npmi_test_std'], digits=4),
    'Topic Diversity (test)': format_mean_std(summary_stats.at[0, 'topic_diversity_mean'], summary_stats.at[0, 'topic_diversity_std'], digits=4),
    'Intra-topic Similarity (test)': format_mean_std(summary_stats.at[0, 'intra_topic_similarity_mean'], summary_stats.at[0, 'intra_topic_similarity_std'], digits=4),
    'Inter-topic Similarity (test)': format_mean_std(summary_stats.at[0, 'inter_topic_similarity_mean'], summary_stats.at[0, 'inter_topic_similarity_std'], digits=4),
    'Silhouette (test, cosine)': format_mean_std(summary_stats.at[0, 'test_silhouette_mean'], summary_stats.at[0, 'test_silhouette_std'], digits=4),
    'Outlier Ratio (test)': format_mean_std(summary_stats.at[0, 'test_outlier_ratio_mean'], summary_stats.at[0, 'test_outlier_ratio_std'], digits=4),
    'Topic Count (test)': format_mean_std(summary_stats.at[0, 'test_topic_count_mean'], summary_stats.at[0, 'test_topic_count_std'], digits=2)
}])

print('\nThesis-ready reporting table (mean ± std across seeds):')
display(thesis_report)

Batches: 100%|██████████| 213/213 [00:05<00:00, 36.11it/s]


Running final fit/eval for seed=42...
Running final fit/eval for seed=7...
Running final fit/eval for seed=123...
Running final fit/eval for seed=2024...
Running final fit/eval for seed=99...
Running final fit/eval for seed=11...
Running final fit/eval for seed=77...
Running final fit/eval for seed=314...
Running final fit/eval for seed=2718...

Per-seed test metrics:


,seed,cv_test,npmi_test,topic_diversity_test,intra_topic_similarity_test,inter_topic_similarity_test,test_silhouette,test_outlier_ratio,test_topic_count
0,42,0.515978,0.013044,0.774832,0.628418,0.296125,0.138285,0.572849,596
1,7,0.505459,0.001884,0.773199,0.621839,0.302005,0.143480,0.586844,597
2,123,0.511996,0.011468,0.776821,0.627074,0.302119,0.138294,0.572702,604
3,2024,0.512968,0.014353,0.781046,0.622387,0.302259,0.146185,0.588760,612
4,99,0.509562,0.009199,0.777419,0.625600,0.306461,0.135842,0.568282,589
5,11,0.509855,0.007169,0.770537,0.620271,0.303192,0.143980,0.574617,577
6,77,0.506026,0.004286,0.781742,0.628026,0.305603,0.136282,0.575280,597
7,314,0.501996,-0.000531,0.779085,0.628117,0.304025,0.136936,0.569387,612
8,2718,0.509040,0.005950,0.774523,0.622388,0.304035,0.141494,0.562758,577



Final test metrics variability across random seeds:


,cv_test_mean,cv_test_std,npmi_test_mean,npmi_test_std,topic_diversity_mean,topic_diversity_std,intra_topic_similarity_mean,intra_topic_similarity_std,inter_topic_similarity_mean,inter_topic_similarity_std,test_silhouette_mean,test_silhouette_std,test_outlier_ratio_mean,test_outlier_ratio_std,test_topic_count_mean,test_topic_count_std
0,0.509209,0.00425,0.007425,0.005064,0.776578,0.003689,0.624902,0.003185,0.302869,0.002965,0.140086,0.003787,0.574609,0.008403,595.666667,12.980755



Thesis-ready reporting table (mean ± std across seeds):


,Model,Seeds (n),Coherence C_v (test),Coherence NPMI (test),Topic Diversity (test),Intra-topic Similarity (test),Inter-topic Similarity (test),"Silhouette (test, cosine)",Outlier Ratio (test),Topic Count (test)
0,BERTopic (best CV config),9,0.5092 ± 0.0042,0.0074 ± 0.0051,0.7766 ± 0.0037,0.6249 ± 0.0032,0.3029 ± 0.0030,0.1401 ± 0.0038,0.5746 ± 0.0084,595.67 ± 12.98


Runs cluster diagnostics and visual sanity-check plots.

In [14]:
import plotly.express as px

# Diagnostics use the currently available final test assignment (`test_topics`) and embeddings (`test_embeddings`).
if 'test_topics' not in globals() or 'test_embeddings' not in globals():
    raise RuntimeError('Please run Cell 24 first so `test_topics` and `test_embeddings` are available.')

labels = np.asarray(test_topics)
emb = np.asarray(test_embeddings)

if labels.shape[0] != emb.shape[0]:
    raise ValueError(f'Mismatch: labels={labels.shape[0]} vs embeddings={emb.shape[0]}')

is_outlier = labels == -1
outlier_ratio_now = float(np.mean(is_outlier)) if len(labels) > 0 else np.nan

valid_labels = labels[~is_outlier]
if len(valid_labels) > 0:
    cluster_size_series = pd.Series(valid_labels).value_counts().sort_values(ascending=False)
else:
    cluster_size_series = pd.Series(dtype='int64')

n_valid_clusters_now = int(cluster_size_series.shape[0])
singleton_topics_now = set(cluster_size_series[cluster_size_series == 1].index.tolist())
n_singletons_now = int(len(singleton_topics_now))

print('Final-clustering diagnostics (current test assignment):')
print(f'  Documents: {len(labels)}')
print(f'  Outlier ratio (-1): {outlier_ratio_now:.4f}')
print(f'  Valid clusters (excluding -1): {n_valid_clusters_now}')
print(f'  Singleton clusters: {n_singletons_now}')

if n_valid_clusters_now == 0:
    print('⚠️ No valid clusters found (all points are outliers).')
elif n_valid_clusters_now == 1:
    print('⚠️ Only one valid cluster found. Silhouette is undefined by design.')
if n_singletons_now > 0:
    print('⚠️ Singleton clusters present. Silhouette excludes them in our robust function.')

# 1) Cluster size distribution (excluding outliers)
if n_valid_clusters_now > 0:
    cluster_size_df = (
        cluster_size_series
        .rename_axis('Topic')
        .reset_index(name='Count')
        .sort_values('Count', ascending=False)
    )
    cluster_size_df['Topic'] = cluster_size_df['Topic'].astype(int)

    fig_sizes = px.bar(
        cluster_size_df,
        x='Topic',
        y='Count',
        title='Final Test Cluster Sizes (excluding outliers)',
        labels={'Topic': 'Topic ID', 'Count': 'Number of Documents'}
    )
    fig_sizes.show()
else:
    print('Cluster-size plot skipped (no valid clusters).')

# 2) 2D embedding view colored by diagnostic point type
# Recompute a 2D projection from test embeddings for visual inspection.
umap_viz = UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
emb_2d = umap_viz.fit_transform(emb)

point_type = np.where(
    labels == -1,
    'Outlier (-1)',
    np.where(np.isin(labels, list(singleton_topics_now)), 'Singleton cluster', 'Valid cluster')
)

viz_df = pd.DataFrame({
    'x': emb_2d[:, 0],
    'y': emb_2d[:, 1],
    'topic': labels.astype(int),
    'point_type': point_type
})

fig_diag = px.scatter(
    viz_df,
    x='x',
    y='y',
    color='point_type',
    hover_data=['topic'],
    title='Final Test Embedding Diagnostics (Outliers / Singleton / Valid)'
)
fig_diag.update_traces(marker=dict(size=6, opacity=0.7))
fig_diag.show()

# 3) Seed-level context from the multi-seed run
if 'seed_results' in globals() and isinstance(seed_results, pd.DataFrame) and not seed_results.empty:
    seed_plot_df = seed_results.copy()

    fig_seed_outliers = px.bar(
        seed_plot_df,
        x='seed',
        y='test_outlier_ratio',
        title='Outlier Ratio by Seed',
        labels={'seed': 'Seed', 'test_outlier_ratio': 'Outlier Ratio'}
    )
    fig_seed_outliers.show()

    fig_seed_topics = px.bar(
        seed_plot_df,
        x='seed',
        y='test_topic_count',
        title='Valid Topic Count by Seed',
        labels={'seed': 'Seed', 'test_topic_count': 'Topic Count (excluding -1)'}
    )
    fig_seed_topics.show()
else:
    print('Seed-level plots skipped (run Cell 24 first to populate `seed_results`).')

Final-clustering diagnostics (current test assignment):
  Documents: 13576
  Outlier ratio (-1): 0.5628
  Valid clusters (excluding -1): 457
  Singleton clusters: 82
⚠️ Singleton clusters present. Silhouette excludes them in our robust function.


Cluster diagnostics and visual sanity checks (outliers, valid clusters, singleton clusters)

## Scientific Tables, Visualizations, and Final Reporting

In [15]:
import plotly.express as px
import plotly.graph_objects as go

if 'df' not in globals():
    raise RuntimeError('Please run the data loading/preparation cells first so `df`, `train_df`, `val_df`, and `test_df` exist.')

if 'summary_stats' not in globals() or 'seed_results' not in globals():
    raise RuntimeError('Please run the multi-seed final evaluation cell first so `summary_stats` and `seed_results` exist.')

if 'tuning_results' not in globals() or 'cv_results' not in globals():
    raise RuntimeError('Please run the hyperparameter tuning cell first so `tuning_results` and `cv_results` exist.')

# ---------- Table 1: Corpus + Split statistics ----------
split_table = pd.DataFrame([
    {
        'Split': 'Train',
        'Documents': len(train_df),
        'Share (%)': 100 * len(train_df) / len(df) if len(df) else np.nan,
        'Start': pd.to_datetime(train_df['date']).min(),
        'End': pd.to_datetime(train_df['date']).max(),
        'Unique Days': train_df['date'].dt.floor('D').nunique()
    },
    {
        'Split': 'Validation',
        'Documents': len(val_df),
        'Share (%)': 100 * len(val_df) / len(df) if len(df) else np.nan,
        'Start': pd.to_datetime(val_df['date']).min(),
        'End': pd.to_datetime(val_df['date']).max(),
        'Unique Days': val_df['date'].dt.floor('D').nunique()
    },
    {
        'Split': 'Test',
        'Documents': len(test_df),
        'Share (%)': 100 * len(test_df) / len(df) if len(df) else np.nan,
        'Start': pd.to_datetime(test_df['date']).min(),
        'End': pd.to_datetime(test_df['date']).max(),
        'Unique Days': test_df['date'].dt.floor('D').nunique()
    }
])

corpus_table = pd.DataFrame([
    {
        'Total Documents': len(df),
        'Unique Headlines': int(df['headline'].nunique()) if 'headline' in df.columns else np.nan,
        'Date Start': pd.to_datetime(df['date']).min(),
        'Date End': pd.to_datetime(df['date']).max(),
        'Total Unique Days': df['date'].dt.floor('D').nunique()
    }
])

print('Table A — Corpus Overview')
display(corpus_table)
print('Table B — Chronological Split Design')
display(split_table)

# ---------- Table 2: Tuning leaderboard (scientific report style) ----------
leaderboard_cols = [
    'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples',
    'cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity',
    'inter_topic_similarity', 'n_topics', 'fit_seconds', 'composite_score'
]

available_cols = [col for col in leaderboard_cols if col in tuning_results.columns]
leaderboard = tuning_results[available_cols].head(10).copy()

for col in ['cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity', 'composite_score', 'fit_seconds']:
    if col in leaderboard.columns:
        leaderboard[col] = pd.to_numeric(leaderboard[col], errors='coerce').round(4)

print('Table C — Top-10 Hyperparameter Configurations (rolling CV)')
display(leaderboard)

# ---------- Table 3: Final test performance (mean ± std across seeds) ----------
final_metrics = pd.DataFrame([
    {
        'Metric': 'Coherence C_v',
        'Mean': summary_stats.at[0, 'cv_test_mean'],
        'Std': summary_stats.at[0, 'cv_test_std']
    },
    {
        'Metric': 'Coherence NPMI',
        'Mean': summary_stats.at[0, 'npmi_test_mean'],
        'Std': summary_stats.at[0, 'npmi_test_std']
    },
    {
        'Metric': 'Topic Diversity',
        'Mean': summary_stats.at[0, 'topic_diversity_mean'],
        'Std': summary_stats.at[0, 'topic_diversity_std']
    },
    {
        'Metric': 'Intra-topic Similarity',
        'Mean': summary_stats.at[0, 'intra_topic_similarity_mean'],
        'Std': summary_stats.at[0, 'intra_topic_similarity_std']
    },
    {
        'Metric': 'Inter-topic Similarity',
        'Mean': summary_stats.at[0, 'inter_topic_similarity_mean'],
        'Std': summary_stats.at[0, 'inter_topic_similarity_std']
    },
    {
        'Metric': 'Silhouette (cosine)',
        'Mean': summary_stats.at[0, 'test_silhouette_mean'],
        'Std': summary_stats.at[0, 'test_silhouette_std']
    },
    {
        'Metric': 'Outlier Ratio',
        'Mean': summary_stats.at[0, 'test_outlier_ratio_mean'],
        'Std': summary_stats.at[0, 'test_outlier_ratio_std']
    },
    {
        'Metric': 'Topic Count',
        'Mean': summary_stats.at[0, 'test_topic_count_mean'],
        'Std': summary_stats.at[0, 'test_topic_count_std']
    }
])

final_metrics['Mean'] = pd.to_numeric(final_metrics['Mean'], errors='coerce').round(4)
final_metrics['Std'] = pd.to_numeric(final_metrics['Std'], errors='coerce').round(4)
final_metrics['Mean ± Std'] = final_metrics.apply(
    lambda row: f"{row['Mean']:.4f} ± {row['Std']:.4f}" if pd.notna(row['Mean']) and pd.notna(row['Std']) else 'NA',
    axis=1
)

print('Table D — Final Test Metrics Across Seeds')
display(final_metrics[['Metric', 'Mean ± Std', 'Mean', 'Std']])

Table A — Corpus Overview


,Total Documents,Unique Headlines,Date Start,Date End,Total Unique Days
0,90048,90048,2014-02-17 05:11:00+00:00,2026-04-11 17:07:00+00:00,3302


Table B — Chronological Split Design


,Split,Documents,Share (%),Start,End,Unique Days
0,Train,62969,69.928260,2014-02-17 05:11:00+00:00,2025-06-16 22:56:30+00:00,3003
1,Validation,13503,14.995336,2025-06-17 02:28:37+00:00,2025-11-18 23:36:34+00:00,155
2,Test,13576,15.076404,2025-11-19 00:09:00+00:00,2026-04-11 17:07:00+00:00,144


Table C — Top-10 Hyperparameter Configurations (rolling CV)


,n_neighbors,n_components,min_cluster_size,min_samples,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,n_topics,fit_seconds,composite_score
0,50,10,20,3,0.5646,0.0721,0.8103,0.5343,0.4052,173.666667,88.6121,0.7410
1,50,10,10,3,0.5607,0.0592,0.8227,0.5732,0.3610,312.666667,88.4856,0.7292
2,50,5,20,3,0.5633,0.0696,0.7995,0.5251,0.4111,171.333333,84.8245,0.6546
3,50,10,15,3,0.5492,0.0541,0.8155,0.5458,0.3956,219.666667,86.6462,0.6331
4,50,5,20,1,0.5618,0.0673,0.7949,0.5178,0.4175,197.000000,85.5163,0.6103
5,15,5,20,3,0.5572,0.0596,0.7990,0.5081,0.4306,214.333333,67.8391,0.5886
6,15,10,20,3,0.5526,0.0574,0.8017,0.5052,0.4330,213.333333,67.5360,0.5718
7,30,5,10,3,0.5512,0.0473,0.8083,0.5553,0.3788,339.333333,76.7924,0.5564
8,30,5,20,3,0.5499,0.0567,0.7997,0.5166,0.4213,186.666667,74.4792,0.5465
9,30,5,15,3,0.5469,0.0486,0.8083,0.5303,0.4067,236.333333,75.4038,0.5465


Table D — Final Test Metrics Across Seeds


,Metric,Mean ± Std,Mean,Std
0,Coherence C_v,0.5092 ± 0.0042,0.5092,0.0042
1,Coherence NPMI,0.0074 ± 0.0051,0.0074,0.0051
2,Topic Diversity,0.7766 ± 0.0037,0.7766,0.0037
3,Intra-topic Similarity,0.6249 ± 0.0032,0.6249,0.0032
4,Inter-topic Similarity,0.3029 ± 0.0030,0.3029,0.0030
5,Silhouette (cosine),0.1401 ± 0.0038,0.1401,0.0038
6,Outlier Ratio,0.5746 ± 0.0084,0.5746,0.0084
7,Topic Count,595.6667 ± 12.9808,595.6667,12.9808


Generates scientific figures for trends, topic prevalence, tuning trade-offs, and stability.

In [16]:
# ---------- Figure 1: Document intensity over time (all splits) ----------
plot_df = df[['date']].copy()
plot_df['date'] = pd.to_datetime(plot_df['date'])
plot_df['day'] = plot_df['date'].dt.floor('D')

daily_counts = plot_df.groupby('day').size().reset_index(name='documents')

split_dates = {
    'train_end': pd.to_datetime(train_df['date']).max(),
    'val_start': pd.to_datetime(val_df['date']).min(),
    'val_end': pd.to_datetime(val_df['date']).max(),
    'test_start': pd.to_datetime(test_df['date']).min()
}

fig_daily = px.line(
    daily_counts,
    x='day',
    y='documents',
    title='Daily Document Volume Over Time',
    labels={'day': 'Date', 'documents': 'Number of Documents'}
)


def add_date_marker(fig, date_value, label, color):
    if pd.isna(date_value):
        return
    marker_x = pd.to_datetime(date_value).to_pydatetime()
    fig.add_shape(
        type='line',
        x0=marker_x,
        x1=marker_x,
        y0=0,
        y1=1,
        xref='x',
        yref='paper',
        line=dict(color=color, dash='dash', width=1.5)
    )
    fig.add_annotation(
        x=marker_x,
        y=1,
        xref='x',
        yref='paper',
        yshift=10,
        text=label,
        showarrow=False,
        font=dict(color=color)
    )


add_date_marker(fig_daily, split_dates['train_end'], 'Train End', 'orange')
add_date_marker(fig_daily, split_dates['val_end'], 'Validation End', 'red')
fig_daily.update_layout(template='plotly_white')
fig_daily.show()

# ---------- Figure 2: Topic prevalence in final test assignment ----------
if 'test_topics' in globals():
    topic_counts = pd.Series(test_topics).value_counts(dropna=False).rename_axis('topic').reset_index(name='count')
    topic_counts['topic'] = topic_counts['topic'].astype(int)
    topic_counts = topic_counts.sort_values('count', ascending=False)

    fig_topic_prev = px.bar(
        topic_counts.head(20),
        x='topic',
        y='count',
        color='topic',
        title='Top 20 Topics by Test-Set Frequency (includes outlier topic -1)',
        labels={'topic': 'Topic ID', 'count': 'Document Count'}
    )
    fig_topic_prev.update_layout(showlegend=False, template='plotly_white')
    fig_topic_prev.show()
else:
    print('Topic prevalence figure skipped (`test_topics` not available).')

# ---------- Figure 3: Hyperparameter trade-off map ----------
if {'cv_val', 'topic_diversity', 'composite_score', 'n_neighbors', 'min_cluster_size'}.issubset(tuning_results.columns):
    fig_tradeoff = px.scatter(
        tuning_results,
        x='cv_val',
        y='topic_diversity',
        color='composite_score',
        size='n_topics' if 'n_topics' in tuning_results.columns else None,
        hover_data=['n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'fit_seconds'],
        title='Tuning Trade-off: Coherence vs Diversity (colored by composite score)',
        labels={'cv_val': 'Validation Coherence (C_v)', 'topic_diversity': 'Topic Diversity', 'composite_score': 'Composite Score'}
    )
    fig_tradeoff.update_layout(template='plotly_white')
    fig_tradeoff.show()

# ---------- Figure 4: Seed stability profile ----------
if not seed_results.empty:
    long_seed = seed_results.melt(
        id_vars=['seed'],
        value_vars=['cv_test', 'npmi_test', 'topic_diversity_test', 'test_silhouette', 'test_outlier_ratio', 'test_topic_count'],
        var_name='metric',
        value_name='value'
    )

    fig_seed_box = px.box(
        long_seed,
        x='metric',
        y='value',
        points='all',
        color='metric',
        title='Final Model Stability Across Random Seeds',
        labels={'metric': 'Metric', 'value': 'Observed Value'}
    )
    fig_seed_box.update_layout(showlegend=False, template='plotly_white')
    fig_seed_box.show()

# ---------- Figure 5: Fold-level metric heatmap ----------
if {'fold', 'cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity'}.issubset(cv_results.columns):
    fold_summary = cv_results.groupby('fold', as_index=False)[
        ['cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity']
    ].mean()

    z_values = fold_summary[['cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity']].to_numpy()

    fig_heat = go.Figure(data=go.Heatmap(
        z=z_values,
        x=['C_v', 'NPMI', 'Diversity', 'Intra Sim', 'Inter Sim'],
        y=[f"Fold {int(f)}" for f in fold_summary['fold']],
        colorscale='Viridis',
        colorbar=dict(title='Mean Value')
    ))
    fig_heat.update_layout(
        title='Fold-wise Average Metrics During Hyperparameter Tuning',
        xaxis_title='Metric',
        yaxis_title='Time-Series CV Fold',
        template='plotly_white'
    )
    fig_heat.show()

print('Scientific visualizations generated.')

Scientific visualizations generated.


## Export reusable outputs (topics, row-level assignments, model results)

In [17]:
from pathlib import Path
from datetime import datetime
import json

required_globals = ['df', 'train_df', 'val_df', 'test_df']
missing = [name for name in required_globals if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required variables: {missing}. Run the upstream pipeline cells first.")

run_tag = datetime.now().strftime('%Y%m%d_%H%M%S')
base_dir = Path('Outputs') / 'GSIB_BERTopic_FinLang/investopedia_embedding' / run_tag
export_dirs = {
    'tables': base_dir / 'tables',
    'row_level': base_dir / 'row_level',
    'models': base_dir / 'models',
    'figures': base_dir / 'figures',
    'meta': base_dir / 'meta'
}
for folder in export_dirs.values():
    folder.mkdir(parents=True, exist_ok=True)

print(f'Export root: {base_dir.resolve()}')


def save_df(df_obj, stem, folder):
    csv_path = folder / f'{stem}.csv'
    df_obj.to_csv(csv_path, index=False, encoding='utf-8-sig')

    parquet_path = folder / f'{stem}.parquet'
    try:
        df_obj.to_parquet(parquet_path, index=False)
        parquet_ok = True
    except Exception:
        parquet_ok = False

    return {
        'csv': str(csv_path),
        'parquet': str(parquet_path) if parquet_ok else None
    }


export_log = {
    'run_tag': run_tag,
    'created_at': datetime.now().isoformat(),
    'paths': {},
    'notes': []
}

# ------------------------------
# 1) Topic-level outputs
# ------------------------------
if 'best_topic_model' in globals():
    active_model = best_topic_model
elif 'topic_model' in globals():
    active_model = topic_model
else:
    active_model = None

if active_model is not None:
    topic_info = active_model.get_topic_info()
    export_log['paths']['topic_info'] = save_df(topic_info, 'topic_info', export_dirs['tables'])

    topic_terms_rows = []
    for topic_id in topic_info['Topic'].tolist():
        if int(topic_id) == -1:
            continue
        terms = active_model.get_topic(int(topic_id))
        if not terms:
            continue
        for rank, (term, weight) in enumerate(terms, start=1):
            topic_terms_rows.append({
                'topic_id': int(topic_id),
                'term_rank': int(rank),
                'term': str(term),
                'weight': float(weight)
            })

    topic_terms_df = pd.DataFrame(topic_terms_rows)
    export_log['paths']['topic_terms'] = save_df(topic_terms_df, 'topic_terms_long', export_dirs['tables'])
else:
    export_log['notes'].append('No BERTopic model found in globals; skipped topic-level exports.')

# ------------------------------
# 2) Row-level topic assignments
# ------------------------------

def build_assignment(split_name, split_df, topics):
    n = min(len(split_df), len(topics))
    if n == 0:
        return pd.DataFrame(columns=['split', 'row_index', 'date', 'headline', 'topic'])

    out_df = split_df.iloc[:n][['date', 'headline']].copy().reset_index(drop=True)
    out_df.insert(0, 'split', split_name)
    out_df.insert(1, 'row_index', np.arange(n))
    out_df['topic'] = pd.Series(topics[:n]).astype('int64')
    return out_df

assignment_frames = []
if 'topics_train' in globals():
    assignment_frames.append(build_assignment('train', train_df, topics_train))
if 'val_topics' in globals():
    assignment_frames.append(build_assignment('validation', val_df, val_topics))
if 'test_topics' in globals():
    assignment_frames.append(build_assignment('test', test_df, test_topics))

if assignment_frames:
    assignments_all = pd.concat(assignment_frames, axis=0, ignore_index=True)
    export_log['paths']['row_level_assignments'] = save_df(assignments_all, 'row_topic_assignments_all', export_dirs['row_level'])

    topic_prevalence = (
        assignments_all
        .groupby(['split', 'topic'], as_index=False)
        .size()
        .rename(columns={'size': 'documents'})
        .sort_values(['split', 'documents'], ascending=[True, False])
    )
    export_log['paths']['topic_prevalence'] = save_df(topic_prevalence, 'topic_prevalence_by_split', export_dirs['tables'])
else:
    export_log['notes'].append('No split-topic arrays found (`topics_train`, `val_topics`, `test_topics`).')

# ------------------------------
# 3) Tuning + evaluation results
# ------------------------------
optional_tables = {
    'cv_results': 'cv_results',
    'tuning_results': 'tuning_results',
    'best_per_fold': 'best_params_per_fold',
    'seed_results': 'seed_results',
    'summary_stats': 'summary_stats',
    'thesis_report': 'thesis_report',
    'final_metrics': 'final_metrics',
    'leaderboard': 'tuning_leaderboard_top10',
    'split_table': 'split_table',
    'corpus_table': 'corpus_table'
}

for var_name, file_stem in optional_tables.items():
    if var_name in globals() and isinstance(globals()[var_name], pd.DataFrame):
        export_log['paths'][var_name] = save_df(globals()[var_name], file_stem, export_dirs['tables'])

if 'best_params' in globals() and isinstance(best_params, dict):
    best_params_path = export_dirs['meta'] / 'best_params.json'
    with open(best_params_path, 'w', encoding='utf-8') as handle:
        json.dump(best_params, handle, indent=2, ensure_ascii=False, default=str)
    export_log['paths']['best_params_json'] = str(best_params_path)

# ------------------------------
# 4) Model artifact (optional)
# ------------------------------
if active_model is not None:
    model_dir = export_dirs['models'] / 'bertopic_model'
    try:
        active_model.save(model_dir, serialization='safetensors', save_ctfidf=True, save_embedding_model=False)
        export_log['paths']['bertopic_model_dir'] = str(model_dir)
    except Exception as model_error:
        export_log['notes'].append(f'Model save skipped: {model_error}')

# ------------------------------
# 5) Save figures as HTML (if available)
# ------------------------------
figure_vars = {
    'fig_daily': 'figure_daily_document_volume',
    'fig_topic_prev': 'figure_topic_prevalence_top20',
    'fig_tradeoff': 'figure_tuning_tradeoff',
    'fig_seed_box': 'figure_seed_stability_box',
    'fig_heat': 'figure_fold_metric_heatmap',
    'fig_diag': 'figure_embedding_diagnostics',
    'fig_seed_outliers': 'figure_seed_outlier_ratio',
    'fig_seed_topics': 'figure_seed_topic_count'
}

for fig_var, fig_name in figure_vars.items():
    fig_obj = globals().get(fig_var)
    if fig_obj is not None and hasattr(fig_obj, 'write_html'):
        fig_path = export_dirs['figures'] / f'{fig_name}.html'
        fig_obj.write_html(fig_path)
        export_log['paths'][fig_var] = str(fig_path)

# ------------------------------
# 6) Export metadata
# ------------------------------
summary_payload = {
    'run_tag': run_tag,
    'export_root': str(base_dir),
    'document_counts': {
        'all': int(len(df)),
        'train': int(len(train_df)),
        'validation': int(len(val_df)),
        'test': int(len(test_df))
    },
    'saved_items': sorted(export_log['paths'].keys()),
    'notes': export_log['notes']
}

summary_path = export_dirs['meta'] / 'export_summary.json'
with open(summary_path, 'w', encoding='utf-8') as handle:
    json.dump(summary_payload, handle, indent=2, ensure_ascii=False)

log_path = export_dirs['meta'] / 'export_log.json'
with open(log_path, 'w', encoding='utf-8') as handle:
    json.dump(export_log, handle, indent=2, ensure_ascii=False)

print('\nSaved artifacts:')
for key in sorted(export_log['paths'].keys()):
    print(f'- {key}')

if export_log['notes']:
    print('\nNotes:')
    for note in export_log['notes']:
        print(f'- {note}')

print(f'\nExport complete. Summary file: {summary_path}')

Export root: C:\Users\gianf\OneDrive - ZHAW\Dokumente\Studium\Masterarbeit\Code\master-thesis\Code\Outputs\GSIB_BERTopic_FinLang\investopedia_embedding\20260423_045921


2026-04-23 04:59:22,293 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`



Saved artifacts:
- best_params_json
- best_per_fold
- corpus_table
- cv_results
- fig_daily
- fig_diag
- fig_heat
- fig_seed_box
- fig_seed_outliers
- fig_seed_topics
- fig_topic_prev
- fig_tradeoff
- final_metrics
- leaderboard
- row_level_assignments
- seed_results
- split_table
- summary_stats
- thesis_report
- topic_info
- topic_prevalence
- topic_terms
- tuning_results

Notes:
- Model save skipped: 'tokenizer'

Export complete. Summary file: Outputs\GSIB_BERTopic_FinLang\investopedia_embedding\20260423_045921\meta\export_summary.json
